In [ ]:
# !pip install langchain chromadb openai sentence-transformers

In [ ]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Setup vectorstore (same as Day 12)
documents = ["Paris is the capital of France. The Eiffel Tower is in Paris.",
             "Jupiter is the largest planet.",
             "Shakespeare wrote Hamlet."]
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.create_documents(documents)
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(chunks, embeddings)

llm = ChatOpenAI(model="gpt-3.5-turbo")

# Multi-query retriever
retriever = MultiQueryRetriever.from_llm(retriever=vectorstore.as_retriever(), llm=llm)
docs = retriever.get_relevant_documents("What is the famous tower in Paris?")
print([d.page_content for d in docs])

In [ ]:
# HyDE implementation
def hyde_retrieve(query, llm, vectorstore, k=3):
    # Generate hypothetical answer
    hypo = llm.predict(f"Write a short passage answering: {query}")
    # Embed the hypothetical answer
    results = vectorstore.similarity_search(hypo, k=k)
    return results

hyde_docs = hyde_retrieve("What is the capital of France?", llm, vectorstore)
print(hyde_docs[0].page_content)

In [ ]:
# Reranking with cross-encoder
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

query = "What is the capital of France?"
initial_docs = vectorstore.similarity_search(query, k=10)
pairs = [[query, doc.page_content] for doc in initial_docs]
scores = cross_encoder.predict(pairs)

# Sort by score
scored_docs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)
top_3 = [doc for doc, score in scored_docs[:3]]
print(top_3[0].page_content)